# **SVD Model**

In [ ]:
import pandas as pd
from surprise import Reader, Dataset, SVD
from surprise.model_selection import GridSearchCV
from IPython.display import display
from tqdm.auto import tqdm
from pathlib import Path
import kagglehub
import time

### **Load Data and Train-Test Split**

##### Load Data

In [2]:
# Download data if it isn't downloaded already

if not Path("books_db/ratings.csv").exists():
    kagglehub.dataset_download("zygmunt/goodbooks-10k", output_dir="books_db")
else:
    print("File already exists")

File already exists


In [3]:
# Read in csvs
file_path = "books_db"
ratings_df = pd.read_csv(f"{file_path}/ratings.csv")
books_df = pd.read_csv(f"{file_path}/books.csv")
display(ratings_df.head())
display(ratings_df.shape)

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4


(981756, 3)

##### Train-Test Split

In [4]:
# This split is kept consistent across models
rating_counts = ratings_df['user_id'].value_counts()
eligible_users = rating_counts[rating_counts >= 10].index
ratings_eligible = ratings_df[ratings_df['user_id'].isin(eligible_users)].copy()

def split_group(group, frac=0.7, seed=42):
    train = group.sample(frac=frac, random_state=seed)
    test = group.drop(train.index)
    return train, test

train_parts = []
test_parts = []

for user_id, group in ratings_eligible.groupby('user_id'):
    train, test = split_group(group)
    train_parts.append(train)
    test_parts.append(test)

ratings_train = pd.concat(train_parts).reset_index(drop=True)
ratings_test = pd.concat(test_parts).reset_index(drop=True)

In [5]:
ratings_train

,book_id,user_id,rating
0,1199,7,4
1,3246,7,4
2,1646,7,3
3,585,7,4
4,4459,7,4
...,...,...,...
599760,7833,53424,4
599761,8213,53424,4
599762,4214,53424,5
599763,9849,53424,4


In [6]:
ratings_test

,book_id,user_id,rating
0,910,7,4
1,956,7,5
2,1801,7,5
3,1923,7,4
4,2129,7,5
...,...,...,...
257768,4483,53424,5
257769,7212,53424,4
257770,7503,53424,4
257771,8262,53424,4


### **Pre-processing**

In [7]:
# add later maybe. double check for missing values, etc.

### **Create Model**

In [8]:
# Give surprise the ratings scale
reader = Reader(rating_scale=(1,5))

# Load data into surprise
dataset = Dataset.load_from_df(ratings_train[['user_id', 'book_id', 'rating']], reader)
trainset = dataset.build_full_trainset()

# Create parameters for grid search (tested others before this)
param_grid = {
    'n_factors': [200, 250],
    'n_epochs': [50, 75],
    'lr_all': [0.02, 0.03],
    'reg_all': [0.1, 0.2]
}

# Set up grid search
grid_search = GridSearchCV(SVD, param_grid, measures=['rmse', 'fcp'], cv=3, n_jobs=-1)

# Fit grid search to dataset and record time taken
print("Grid search is running...")
start_time = time.time()
grid_search.fit(dataset)
end_time = time.time()
print(f"Grid Serach Time Taken: {(end_time - start_time):.2f} seconds")

# Print best params (rmse and fcp, fcp because final evaluation will be based on ranking)
print(f"Best RMSE Parameters: {grid_search.best_params['rmse']}")
print(f"Best RMSE Score: {grid_search.best_score['rmse']:.5f}")
print(f"Best FCP Parameters: {grid_search.best_params['fcp']}")
print(f"Best FCP Score: {grid_search.best_score['fcp']:.5f}")

Grid search is running...
Grid Serach Time Taken: 189.93 seconds
Best RMSE Parameters: {'n_factors': 250, 'n_epochs': 75, 'lr_all': 0.02, 'reg_all': 0.1}
Best RMSE Score: 0.83268
Best FCP Parameters: {'n_factors': 250, 'n_epochs': 75, 'lr_all': 0.03, 'reg_all': 0.1}
Best FCP Score: 0.61350


In [9]:
# Choosing fcp because the evaluation across all models will be based on ranking
optimal_params = grid_search.best_params['fcp']

# Train Model
svd = SVD(**optimal_params)
svd.fit(trainset)

### **Evaluation**

<h5 style="margin-bottom: 0;">Top K Recommendations (see if user read/rated the top k chosen by the model)</h5>
<h6 style="margin-top: 0;">Note: changed 'canonical_id' to 'book_id'</h6>

In [10]:
# Evaluation function for comparison with other models taken from content filtering script

def compute_metrics(recommendations, test_ids, k):
    """Compare one user's recommendation df against their held-out test ids."""
    if recommendations.empty or not test_ids:
        return None

    is_hit = recommendations['book_id'].isin(test_ids)
    hits = is_hit.sum()

    precision_at_k = hits / len(recommendations)
    recall_at_k = hits / len(test_ids)

    if (precision_at_k + recall_at_k) > 0:
        f1_at_k = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)
    else:
        f1_at_k = 0.0

    if hits > 0:
        first_hit_rank = is_hit.values.argmax() + 1
        reciprocal_rank = 1 / first_hit_rank
    else:
        reciprocal_rank = 0.0

    return {
        'precision': precision_at_k,
        'recall': recall_at_k,
        'f1': f1_at_k,
        'reciprocal_rank': reciprocal_rank,
    }


def evaluate_model(get_recommendations_fn, eval_users, ratings_test, k=5):
    """Runs any model's recommend function against a set of users and aggregates metrics."""
    precisions= []
    recalls = []
    f1s = []
    reciprocal_ranks = []

    for user_id in eval_users:
        recommendations = get_recommendations_fn(user_id, k)
        user_test_ids = set(ratings_test[ratings_test['user_id'] == user_id]['book_id'])

        metrics = compute_metrics(recommendations, user_test_ids, k)
        if metrics is None:
            continue

        precisions.append(metrics['precision'])
        recalls.append(metrics['recall'])
        f1s.append(metrics['f1'])
        reciprocal_ranks.append(metrics['reciprocal_rank'])

    if not precisions:
        return {'precision': 0, 'recall': 0, 'f1': 0, 'mrr': 0, 'n_users': 0}

    return {
        'precision': sum(precisions) / len(precisions),
        'recall': sum(recalls) / len(recalls),
        'f1': sum(f1s) / len(f1s),
        'mrr': sum(reciprocal_ranks) / len(reciprocal_ranks),
        'n_users': len(precisions),
    }

In [ ]:
all_books = books_df['book_id'].unique()

def svd_k_ranking(user_id, k=5):
    """Uses SVD model to find the top K book recommendations per user"""
    # Find books user has rated
    user_ratings = ratings_train[ratings_train['user_id'] == user_id]
    rated_books = user_ratings['book_id'].tolist()

    # Find books user HASN'T rated
    unrated_books = [book for book in all_books if book not in rated_books]

    # Predict rating for every unrated book
    predictions = []
    for book in unrated_books:
        predicted_score = svd.predict(user_id, book).est

        predictions.append({
            'book_id': book,
            'score': predicted_score
        })

    predictions_df = pd.DataFrame(predictions)
    top_k_recs = predictions_df.sort_values(by="score", ascending=False).head(k)

    return top_k_recs

In [ ]:
# Evaluate using evaluate function, and then print metrics
k = 5
eval_users = ratings_train['user_id'].unique()
test_users = eval_users[:500]

# try and add tqdm

results = evaluate_model(
    get_recommendations_fn=lambda user_id, k: svd_k_ranking(user_id, k),
    eval_users=test_users,
    ratings_test=ratings_test,
    k=k
)

print(f"evaluated {results['n_users']} users")
print(f"Precision@{k}: {results['precision']:.2%}")
print(f"Recall@{k}: {results['recall']:.2%}")
print(f"F1@{k}: {results['f1']:.2%}")
print(f"MRR: {results['mrr']:.4f}")

evaluated 500 users
Precision@5: 0.24%
Recall@5: 0.13%
F1@5: 0.16%
MRR: 0.0072
